In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder
import pickle

In [7]:
path="C:\MAIN\codenotes\AIML\project1\dataset\Churn_Modelling.csv"

<>:1: SyntaxWarning: invalid escape sequence '\M'
<>:1: SyntaxWarning: invalid escape sequence '\M'
C:\Users\kesha\AppData\Local\Temp\ipykernel_12280\2187790677.py:1: SyntaxWarning: invalid escape sequence '\M'
  path="C:\MAIN\codenotes\AIML\project1\dataset\Churn_Modelling.csv"


In [8]:
df=pd.read_csv(path)

In [9]:
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [10]:
#preprocessing

df=df.drop(['RowNumber','CustomerId','Surname'],axis=1)
df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [11]:
encodegender=LabelEncoder()

df['Gender']=encodegender.fit_transform(df['Gender'])
df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0


In [12]:
from sklearn.preprocessing import OneHotEncoder

encoder=OneHotEncoder()
encodedgeo=encoder.fit_transform(df[['Geography']])

encodedgeo


<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 10000 stored elements and shape (10000, 3)>

In [13]:
encoder.get_feature_names_out(['Geography'])

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [14]:
encoded_df=pd.DataFrame(encodedgeo.toarray(),columns=encoder.get_feature_names_out(['Geography']))

In [15]:
#combine encoded columns with original dataframe

df=pd.concat([df.drop('Geography',axis=1),encoded_df],axis=1)

In [16]:
df.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [17]:
#saving pkl file 

with open('encodegender.pkl', 'wb') as file:
    pickle.dump(encodegender, file)
    
with open('encoder.pkl','wb') as file:
    pickle.dump(encoder, file)

In [18]:
#dividing dataset

x=df.drop('Exited',axis=1)
y=df['Exited']

##splitting dataset

x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

#scalling
scaler=StandardScaler()
x_train=scaler.fit_transform(x_train)
x_test=scaler.transform(x_test)


In [19]:
with open('scaler.pkl','wb') as file:
    pickle.dump(scaler,file)

ANN Implementation

In [20]:


import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

In [21]:
x_train.shape[1]

12

In [23]:
#Building model

model=Sequential([
    Dense(64,activation='relu',input_shape=(x_train.shape[1],)),  ##1st hidden layer=input layer
    Dense(32,activation='relu'),     ##hiddenlayer2
    Dense(1,activation='sigmoid')
    
    
])

c:\MAIN\codenotes\AIML\project1\venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [25]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [28]:
opt=tf.keras.optimizers.Adam(learning_rate=0.01)
loss=tf.keras.losses.BinaryCrossentropy()

In [ ]:
#model compilation

model.compile(optimizer=opt,loss=loss,metrics=['accuracy'])

In [54]:
#setup tensorboard

log_dir="logs/fit"+datetime.datetime.now().strftime("%Y%m%d%H%M%S")

In [55]:
tf_callback=TensorBoard(log_dir=log_dir,histogram_freq=1)

In [56]:
##setup for early stopping

earlystoppingcallback=EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True)



In [57]:
##TRAINING MODEL

history=model.fit(
    x_train,y_train,validation_data=(x_test,y_test),epochs=100,
    callbacks=[tf_callback,earlystoppingcallback]
    
)

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8641 - loss: 0.3251 - val_accuracy: 0.8600 - val_loss: 0.3530
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8654 - loss: 0.3237 - val_accuracy: 0.8610 - val_loss: 0.3556
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8660 - loss: 0.3230 - val_accuracy: 0.8645 - val_loss: 0.3472
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8664 - loss: 0.3187 - val_accuracy: 0.8575 - val_loss: 0.3542
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8679 - loss: 0.3175 - val_accuracy: 0.8615 - val_loss: 0.3455
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8677 - loss: 0.3137 - val_accuracy: 0.8475 - val_loss: 0.3683
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8689 - loss: 0.3154 - val_accuracy: 0.8505 - val_loss: 0.3574
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8699 - loss: 0.3115 - val_accu

In [58]:
model.save('model.h5')

In [59]:
##LOAD TENSORBOARD EXTENTSION

%reload_ext tensorboard

In [62]:
%tensorboard --logdir logs/fit20260414010609

Reusing TensorBoard on port 6007 (pid 20412), started 0:00:02 ago. (Use '!kill 20412' to kill it.)